In [2]:
# !pip install -r requirements.txt
# # !pip uninstall -y opencv-python opencv-python-headless numpy
# # !pip uninstall -y numpy  # run twice on purpose (handles duplicates)
# # !pip install --upgrade pip setuptools wheel
# # !pip install numpy==1.26.4
# !pip install opencv-python-headless==4.9.0.80
!pip install lxml
# # !pip install -U xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 32.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [17]:
import os

root_dir = "test_annotations"  # change this to your main directory

for root, dirs, files in os.walk(root_dir):
    for filename in files:
        if filename.startswith(("train_", "test_")):
            old_path = os.path.join(root, filename)

            # Remove prefix (split only on first underscore)
            new_filename = filename.split("_", 1)[1]
            new_path = os.path.join(root, new_filename)

            os.rename(old_path, new_path)
            print(f"Renamed: {old_path} -> {new_path}")

print("Done!")


Renamed: test_annotations/street/test_sun_akoxkqsqbcldtqfz.xml -> test_annotations/street/sun_akoxkqsqbcldtqfz.xml
Renamed: test_annotations/street/train_sun_apyfnhyfvypuwjsx.xml -> test_annotations/street/sun_apyfnhyfvypuwjsx.xml
Renamed: test_annotations/street/test_sun_argnolquzccabeda.xml -> test_annotations/street/sun_argnolquzccabeda.xml
Renamed: test_annotations/street/train_sun_atzyjadgkypylonu.xml -> test_annotations/street/sun_atzyjadgkypylonu.xml
Renamed: test_annotations/street/test_sun_ajzgbgbiyesaqkgn.xml -> test_annotations/street/sun_ajzgbgbiyesaqkgn.xml
Renamed: test_annotations/street/test_sun_apbemvggofwficuo.xml -> test_annotations/street/sun_apbemvggofwficuo.xml
Renamed: test_annotations/street/test_sun_aqkxuykoiwjqfcof.xml -> test_annotations/street/sun_aqkxuykoiwjqfcof.xml
Renamed: test_annotations/street/test_sun_afhyghbzboprduvr.xml -> test_annotations/street/sun_afhyghbzboprduvr.xml
Renamed: test_annotations/street/test_sun_axzfxbfqrkqhgcjy.xml -> test_annotat

In [14]:
import os, re, json, hashlib
from pathlib import Path
import pandas as pd
from PIL import Image
import numpy as np
import cv2
from lxml import etree

import torch
from diffusers import (
    StableDiffusionControlNetImg2ImgPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
)

2026-02-12 04:21:58.681589: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-12 04:21:58.717932: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-12 04:21:58.717974: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-12 04:21:58.718875: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-12 04:21:58.724835: I tensorflow/core/platform/cpu_feature_guar

In [15]:
import cv2
print(cv2.__version__)

4.9.0


In [18]:
# -------------------------
# PATHS (edit these)
# -------------------------
IMAGES_ROOT = Path("images/test")   # e.g., .../train_images
XML_ROOT    = Path("test_annotations")      # e.g., .../train_xml
REQUIRED_CSV = Path("verification_summary_1 (test).csv")  # contains required object names

OUT_DIR = Path("fake_output_test_img_with_text")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_IMG_DIR = OUT_DIR / "images"
OUT_IMG_DIR.mkdir(exist_ok=True)

LOG_CSV = OUT_DIR / "generation_log.csv"

# -------------------------
# MODEL CONFIG
# -------------------------
SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
CONTROLNET_ID = "lllyasviel/control_v11f1p_sd15_depth"  # ControlNet Depth (SD1.5)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# -------------------------
# GENERATION PARAMS (start here)
# -------------------------
NEG_PROMPT = "text, watermark, logo, extra objects not listed, duplicate objects, distorted, low quality, blurry"
NUM_STEPS = 30
CFG = 12.0
STRENGTH = 0.35                 # img2img denoise strength
CONTROLNET_SCALE = 0.85         # how strongly depth guides

# object list cap (optional objects can be added after required ones)
MAX_TOTAL_OBJECTS = 20

In [19]:
# Load required objects from CSV

def normalize_required_object(name: str):
    """
    Rules:
    - split on '_and_' → multiple objects
    - replace '_' with ' '
    """
    name = name.strip()
    parts = name.split("_and_")   # rule 1
    cleaned = [p.replace("_", " ").strip() for p in parts]  # rule 2
    return cleaned

# ---- Load and normalize required objects CSV ----
OBJ_COL = "ecii_concepts"   # adjust if needed

req_df = pd.read_csv(REQUIRED_CSV)

required_set = set()
for raw in req_df[OBJ_COL].dropna().astype(str):
    for obj in normalize_required_object(raw):
        if obj:  # safety
            required_set.add(obj)

print("Required objects after normalization:", len(required_set))
print(sorted(required_set))

Required objects after normalization: 21
['bathtube', 'bidet', 'bouquet', 'bridge', 'building', 'car', 'cars', 'central', 'coffee', 'desk', 'fence', 'fruit bowl', 'shower curtain', 'sideboard', 'sky', 'skyscraper', 'snowy mountain', 'soupdish', 'telephone', 'toilet', 'wardrobe']


In [20]:
# XML parsing utilities (objects + folder/class)

def strip_crop(name: str) -> str:
    # Only rule you requested:
    # "bath crop" -> "bath"
    return re.sub(r"\s+crop\s*$", "", name.strip())

def parse_xml_objects_and_folder(xml_path: Path):
    """
    Returns:
      folder_str: the <folder> text (or None)
      objects: list of normalized object names (crop removed), may contain duplicates
    """
    tree = etree.parse(str(xml_path))
    root = tree.getroot()

    folder_el = root.find("folder")
    folder_str = folder_el.text.strip() if folder_el is not None and folder_el.text else None

    objs = []
    for obj_el in root.findall("object"):
        name_el = obj_el.find("name")
        if name_el is None or not name_el.text:
            continue
        objs.append(strip_crop(name_el.text))

    return folder_str, objs

In [21]:
# Map image → xml 
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def image_to_xml_path(img_path: Path) -> Path:
    rel = img_path.relative_to(IMAGES_ROOT)
    return (XML_ROOT / rel).with_suffix(".xml")

In [38]:
from PIL import Image

def preprocess_for_sd(img: Image.Image) -> tuple[Image.Image, str]:
    """
    Returns (processed_image, mode)
    mode ∈ {"resize_low", "force_512", "keep"}
    """
    img = img.convert("RGB")
    w, h = img.size

    # (1) low-res: both dims < 300 → resize only (no padding)
    if w < 300 or h < 300:
        return img.resize((300, 300), resample=Image.BICUBIC), "resize_low"

    # (2) large: any dim > 512 → force 512×512
    if w > 512 or h > 512:
        return img.resize((512, 512), resample=Image.BICUBIC), "force_512"

    # (3) otherwise keep
    return img, "keep"

In [39]:
# Choose class_name + build prompt objects


def extract_class_name(img_path: Path, folder_str_from_xml) -> str:
    if folder_str_from_xml:
        # take last segment after slash
        return folder_str_from_xml.strip("/").split("/")[-1]
    # fallback: folder name containing the image
    return img_path.parent.name

def build_object_list_for_prompt(xml_objects: list[str], required_set: set[str], max_total: int):
    """
    Ensures: any object present in BOTH xml_objects and required_set is included.
    No synonymization. Only 'crop' stripping already done.
    """
    # unique but stable order (preserve first occurrence)
    seen = set()
    xml_unique = []
    for o in xml_objects:
        if o not in seen:
            seen.add(o)
            xml_unique.append(o)

    required_for_image = [o for o in xml_unique if o in required_set]
    optional = [o for o in xml_unique if o not in required_set]

    # Fill optional up to cap (after required)
    remaining = max(0, max_total - len(required_for_image))
    optional_kept = optional[:remaining]

    final_objs = required_for_image + optional_kept
    return required_for_image, optional_kept, final_objs

In [40]:
# build prompt

def make_prompt(class_name: str, objs: list[str]) -> str:
    # Option B
    obj_str = ", ".join(objs) if objs else ""
    if obj_str:
        return (
            "Generate an image in the style of the given image. "
            f"It should depict a {class_name} scene and must contain: {obj_str}."
        )
    else:
        # fallback if no objs
        return (
            "Generate an image in the style of the given image. "
            f"It should depict a {class_name} scene."
        )

In [41]:
def make_depth_proxy(pil_img: Image.Image) -> Image.Image:
    """
    Placeholder depth proxy: converts to grayscale + smooths + normalizes.
    If you want real MiDaS depth, tell me and I'll swap this.
    """
    img = np.array(pil_img.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (9, 9), 0)
    norm = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)
    depth3 = cv2.cvtColor(norm.astype(np.uint8), cv2.COLOR_GRAY2RGB)
    return Image.fromarray(depth3)

In [42]:
# Load the SD+ControlNet pipeline

from diffusers import StableDiffusionImg2ImgPipeline, UniPCMultistepScheduler

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    SD_MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEVICE)

print("Pipeline loaded on:", DEVICE)

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.9/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion_img2img.StableDiffusionImg2ImgPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look

Pipeline loaded on: cuda


In [43]:
def seed_from_string(s: str) -> int:
    # stable integer seed from filename / path
    h = hashlib.sha256(s.encode("utf-8")).hexdigest()
    return int(h[:8], 16)  # 32-bit-ish

In [44]:
rows = []
img_paths = [p for p in IMAGES_ROOT.rglob("*") if p.suffix.lower() in IMAGE_EXTS]

print("Images found:", len(img_paths))

for idx, img_path in enumerate(sorted(img_paths)):
    xml_path = image_to_xml_path(img_path)
    if not xml_path.exists():
        print(f"[WARN] Missing XML for {img_path} -> {xml_path}")
        continue

    # read image
    init_img = Image.open(img_path).convert("RGB")
    # init_img = cap_to_512(init_img)
    orig_w, orig_h = init_img.size

    init_img, resize_mode = preprocess_for_sd(init_img)

    # parse xml
    folder_str, xml_objs = parse_xml_objects_and_folder(xml_path)
    class_name = extract_class_name(img_path, folder_str)

    # build object list with required guarantee
    req_objs, opt_objs, final_objs = build_object_list_for_prompt(
        xml_objects=xml_objs,
        required_set=required_set,
        max_total=MAX_TOTAL_OBJECTS
    )

    prompt = make_prompt(class_name, final_objs)

    # control image (depth proxy for now)
    control_img = make_depth_proxy(init_img)

    # seed
    seed = seed_from_string(str(img_path.relative_to(IMAGES_ROOT)))
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    # generate
    out = pipe(
    prompt=prompt,
    negative_prompt=NEG_PROMPT,
    image=init_img,
    strength=STRENGTH,
    num_inference_steps=NUM_STEPS,
    guidance_scale=CFG,
    generator=generator,
)
    fake_img = out.images[0]

    # save output with same relative path
    rel = img_path.relative_to(IMAGES_ROOT)
    out_path = (OUT_IMG_DIR / rel).with_suffix(".png")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fake_img.save(out_path)

    rows.append({
        "idx": idx,
        "image_path": str(img_path),
        "xml_path": str(xml_path),
        "out_path": str(out_path),
        "class_name": class_name,
        "xml_folder": folder_str if folder_str else "",
        "required_objs_in_prompt": ";".join(req_objs),
        "optional_objs_in_prompt": ";".join(opt_objs),
        "all_objs_in_prompt": ";".join(final_objs),
        "prompt": prompt,
        "negative_prompt": NEG_PROMPT,
        "seed": seed,
        "strength": STRENGTH,
        "num_steps": NUM_STEPS,
        "cfg": CFG,
        "controlnet_scale": CONTROLNET_SCALE,
        "sd_model": SD_MODEL_ID,
        "controlnet_model": CONTROLNET_ID,
    })

    if (idx + 1) % 25 == 0:
        print(f"Done {idx+1}/{len(img_paths)}")

# write log
log_df = pd.DataFrame(rows)
log_df.to_csv(LOG_CSV, index=False)
print("Saved log:", LOG_CSV, "rows:", len(log_df))

Images found: 793


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

...

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

Saved log: fake_output_test_img_with_text/generation_log.csv rows: 793


In [46]:
import os
k = 0
for i in os.listdir("fake_output_test_img_with_text/images"):
    d = "fake_output_test_img_with_text/images/"+i
    k+=len(os.listdir(d))
k

793

In [47]:
# --- Get activations for ALL fake images under out_fake/<class>/<image> ---

import os, csv
from pathlib import Path
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras.models import load_model

OUT_FAKE_ROOT   = Path("fake_output_test_img_with_text/images")                      # input root with fakes
MODEL_PATH      = "model_INCEPTIONV3_sun10_jul23.h5"    # your .h5 model
OUT_CSV_PATH    = "[img+obj_labels_to_fakes]fake_activations(test).csv"                # output CSV

# 1) Load model once and build penultimate (assumes last-2 is Dense(64))
model = load_model(MODEL_PATH, compile=False)
penultimate = tf.keras.Model(inputs=model.input, outputs=model.layers[-2].output)

# 2) Infer input size (H, W, C)
in_shape = model.input_shape
if isinstance(in_shape, list):
    in_shape = in_shape[0]
_, H_in, W_in, C_in = in_shape

# 3) Prepare CSV
fieldnames = ["filenames"] + [str(i) for i in range(64)]
with open(OUT_CSV_PATH, "w", newline="", encoding="utf-8") as fcsv:
    writer = csv.DictWriter(fcsv, fieldnames=fieldnames)
    writer.writeheader()

    # Walk class folders
    for cls_dir in sorted([d for d in OUT_FAKE_ROOT.iterdir() if d.is_dir()]):
        cls = cls_dir.name
        for img_path in sorted(cls_dir.glob("*.*")):
            if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue

            rel_filename = f"{cls}/{img_path.name}"

            # Load & preprocess
            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize((W_in, H_in), Image.BICUBIC)
                x = tf.keras.utils.img_to_array(img).astype("float32")
                if C_in == 1:
                    x = tf.image.rgb_to_grayscale(x)
                x = x / 255.0
                x = np.expand_dims(x, 0)  # (1,H,W,C)
            except Exception as e:
                print(f"[load/prep] {rel_filename}: {e}")
                continue

            # Forward pass to penultimate
            try:
                act = penultimate.predict(x, verbose=0)[0]  # shape (64,)
            except Exception as e:
                print(f"[predict] {rel_filename}: {e}")
                continue

            # Write one row
            row = {"filenames": rel_filename}
            for j in range(64):
                row[str(j)] = float(act[j])
            writer.writerow(row)

print(f"[done] wrote activations for fakes -> {OUT_CSV_PATH}")


[done] wrote activations for fakes -> [img+obj_labels_to_fakes]fake_activations(test).csv


In [52]:
# --- Step 7: fired-counts utilities (stdlib CSV + numpy) ---
import csv
import numpy as np
import os

# minimal error log (list of dicts); use if you want to capture issues
ERRORS = []
def log_error(stage, item, exc):
    msg = f"[{stage}] {item}: {type(exc).__name__}: {exc}"
    print(msg)
    ERRORS.append({"stage": stage, "item": str(item), "error": f"{type(exc).__name__}: {exc}"})

def load_csv_as_dicts(path):
    with open(path, "r", newline="", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        rows = [r for r in rdr]
        header = rdr.fieldnames
    return header, rows

def find_filename_column(header):
    for cand in ("filenames", "filename", "file", "path"):
        if cand in header:
            return cand
    raise RuntimeError("No filename column found (looked for: filenames/filename/file/path)")

def ensure_neuron_columns(header, n=64):
    cols = [str(i) for i in range(n)]
    missing = [c for c in cols if c not in header]
    if missing:
        raise RuntimeError(f"Missing neuron columns in CSV, e.g., {missing[:5]}")
    return cols

def compute_threshold_from_real(real_rows, neuron_cols, ratio=0.8):
    # Per-neuron max across *all real* rows, then 0.8×
    first = True
    max_act = None
    for r in real_rows:
        vals = np.array([float(r[c]) for c in neuron_cols], dtype=np.float64)
        if first:
            max_act = vals
            first = False
        else:
            max_act = np.maximum(max_act, vals)
    if max_act is None:
        raise RuntimeError("Real CSV had no rows.")
    return ratio * max_act  # numpy array shape (64,)

def count_fired(vec, threshold):
    # vec, threshold are np arrays shape (64,)
    return int(np.sum(vec >= threshold))


def _candidate_fake_keys(orig_fname):
    """
    Given e.g. 'living_room/sun_abc.jpg', try keys commonly seen in fake CSVs:
      1) same path, basename prefixed: 'living_room/fake_sun_abc.jpg'
      2) whole relpath prefixed:       'fake_living_room/sun_abc.jpg' (rare but try)
      3) just basename prefixed:       'fake_sun_abc.jpg'
      4) exact original (in case CSV already matches)
    """
    d = os.path.dirname(orig_fname)
    b = os.path.basename(orig_fname)
    pref_b = f"fake_{b}"

    cand = []
    # same dir, prefixed basename
    cand.append(os.path.join(d, pref_b) if d else pref_b)
    # whole relpath prefixed (least common, but cheap to try)
    cand.append(f"fake_{orig_fname}")
    # just basename prefixed
    cand.append(pref_b)
    # exact original (fallback)
    cand.append(orig_fname)
    # dedupe while preserving order
    seen = set(); out = []
    for c in cand:
        if c not in seen:
            seen.add(c); out.append(c)
    return out

def write_fired_counts(real_csv_path, fake_csv_path, out_counts_csv_path, error_list=None):
    """
    Reads real & fake activations CSVs, computes per-neuron 0.8×max threshold from real,
    and writes rows: filenames,real_count,fake_count
    """
    real_header, real_rows = load_csv_as_dicts(real_csv_path)
    fake_header, fake_rows = load_csv_as_dicts(fake_csv_path)

    real_fname_col = find_filename_column(real_header)
    fake_fname_col = find_filename_column(fake_header)

    neuron_cols = ensure_neuron_columns(real_header, n=64)

    # thresholds from real data
    threshold = compute_threshold_from_real(real_rows, neuron_cols, ratio=0.8)

    # index fake rows by their filename column
    fake_idx = { r[fake_fname_col]: r for r in fake_rows }

    with open(out_counts_csv_path, "w", newline="", encoding="utf-8") as f_out:
        w = csv.writer(f_out)
        w.writerow(["filenames", "real_count", "fake_count"])

        for r in real_rows:
            fname = r[real_fname_col]

            # real count
            try:
                real_vec = np.array([float(r[c]) for c in neuron_cols], dtype=np.float64)
                real_count = count_fired(real_vec, threshold)
            except Exception as e:
                if error_list is not None:
                    error_list.append({"stage":"count-real","item":fname,"error":f"{type(e).__name__}: {e}"})
                real_count = ""

            # fake count (match by filename with fake_ prefix heuristics)
            try:
                fr = None
                for key in _candidate_fake_keys(fname):
                    fr = fake_idx.get(key)
                    if fr is not None:
                        break
                if fr is None:
                    if error_list is not None:
                        error_list.append({"stage":"match-fake","item":fname,"error":"FileNotFoundError: fake row not found (tried fake_<basename> in-place, fake_<relpath>, and basename)"})
                    fake_count = ""
                else:
                    fake_vec = np.array([float(fr[c]) for c in neuron_cols], dtype=np.float64)
                    fake_count = count_fired(fake_vec, threshold)
            except Exception as e:
                if error_list is not None:
                    error_list.append({"stage":"count-fake","item":fname,"error":f"{type(e).__name__}: {e}"})
                fake_count = ""

            w.writerow([fname, real_count, fake_count])

    print(f"[done] wrote fired counts -> {out_counts_csv_path}")



In [53]:
import pandas as pd

# Load CSV
df = pd.read_csv("[img+obj_labels_to_fakes]fake_activations(test).csv")

# Replace .png with .jpg in the 'filenames' column
df["filenames"] = df["filenames"].str.replace(".png", ".jpg", regex=False)

# Save back to CSV
df.to_csv("[img+obj_labels_to_fakes]fake_activations(test).csv", index=False)


In [54]:
# --- Step 8: run the comparison (0.8× rule) ---

REAL_ACT_CSV_PATH = "preds_of_64Neurons_denseLayer_test.csv"  # your file 1 (real)
FAKE_ACT_CSV_PATH = "[img+obj_labels_to_fakes]fake_activations(test).csv"                        # produced by process_class()
FIRED_COUNTS_CSV  = "[img+obj_labels_to_fakes]fired_counts_real_vs_fake(test).csv"               # output (3 cols)

# Reuse the same ERRORS list from earlier if you want to collect issues:
write_fired_counts(REAL_ACT_CSV_PATH, FAKE_ACT_CSV_PATH, FIRED_COUNTS_CSV, error_list=ERRORS)

# (optional) persist errors
if ERRORS:
    with open("pipeline_errors.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["stage","item","error"])
        w.writeheader(); w.writerows(ERRORS)
    print(f"[errors] wrote pipeline_errors.csv with {len(ERRORS)} entries")
else:
    print("[ok] no errors logged")


[done] wrote fired counts -> [img+obj_labels_to_fakes]fired_counts_real_vs_fake(test).csv
[ok] no errors logged


## statistical testing

In [55]:
## for 64 neurons

# Wilcoxon (one-sided, paired) for: H_A = real_confirmed_count > fake_confirmed_count
# Requires: pip install scipy

import csv
import numpy as np
from scipy.stats import wilcoxon, rankdata

CSV_PATH = "[img+obj_labels_to_fakes]fired_counts_real_vs_fake(test).csv"  # filenames, real_confirmed_count, fake_confirmed_count

# --- load paired counts ---
reals, fakes = [], []
with open(CSV_PATH, "r", newline="", encoding="utf-8") as f:
    rdr = csv.DictReader(f)
    for r in rdr:
        try:
            rv = float(r["real_count"])
            fv = float(r["fake_count"])
        except Exception:
            continue
        if np.isnan(rv) or np.isnan(fv):
            continue
        reals.append(rv); fakes.append(fv)

reals = np.asarray(reals, dtype=float)
fakes = np.asarray(fakes, dtype=float)
if reals.size == 0:
    raise RuntimeError("No valid (real,fake) pairs found in CSV. Check column names/values.")

diffs = reals - fakes  # test if diffs > 0  (i.e., fake < real)

# --- run Wilcoxon (zeros dropped by 'zero_method=wilcox') ---
stat, p = wilcoxon(diffs, alternative="greater", zero_method="wilcox", correction=False, mode="auto")

# --- descriptive stats ---
n_total   = diffs.size
nz_mask   = diffs != 0
n_nonzero = int(nz_mask.sum())
n_zero    = n_total - n_nonzero

med_real  = float(np.median(reals))
med_fake  = float(np.median(fakes))
med_diff  = float(np.median(diffs))
mean_diff = float(np.mean(diffs))
prop_pos  = float(np.mean(diffs > 0.0))  # proportion of pairs with real > fake (over all pairs)

# --- effect size: rank-biserial correlation (r_rb), on nonzero diffs only ---
if n_nonzero > 0:
    d_nz = diffs[nz_mask]
    ranks = rankdata(np.abs(d_nz), method="average")       # ranks of absolute diffs
    W_plus  = ranks[d_nz > 0].sum()
    W_minus = ranks[d_nz < 0].sum()
    denom = n_nonzero * (n_nonzero + 1) / 2.0
    r_rb = (W_plus - W_minus) / denom if denom > 0 else np.nan
else:
    r_rb = np.nan

# --- print report ---
print("=== Wilcoxon Signed-Rank Test (one-sided, real > fake) ===")
print(f"File: {CSV_PATH}")
print(f"N total pairs      : {n_total}")
print(f"N nonzero pairs    : {n_nonzero} (pairs used by Wilcoxon)")
print(f"N zero differences : {n_zero}")
print()
print(f"Median(real)       : {med_real:.3f}")
print(f"Median(fake)       : {med_fake:.3f}")
print(f"Median(diff)       : {med_diff:.3f}   (diff = real - fake)")
print(f"Mean(diff)         : {mean_diff:.3f}")
print(f"P(real > fake)     : {prop_pos:.3f}   (fraction of pairs with diff > 0 over all pairs)")
print()
print(f"Wilcoxon statistic : {stat:.6g}")
print(f"One-sided p-value  : {p:.6g}   (H_A: real > fake)")
print(f"Effect size r_rb   : {r_rb:.3f}   (rank-biserial correlation; -1..+1, >0 favors real)")
print("Decision (alpha=0.01): " + ("REJECT H0" if p < 0.05 else "fail to reject H0"))


=== Wilcoxon Signed-Rank Test (one-sided, real > fake) ===
File: [img+obj_labels_to_fakes]fired_counts_real_vs_fake(test).csv
N total pairs      : 793
N nonzero pairs    : 350 (pairs used by Wilcoxon)
N zero differences : 443

Median(real)       : 0.000
Median(fake)       : 0.000
Median(diff)       : 0.000   (diff = real - fake)
Mean(diff)         : 0.372
P(real > fake)     : 0.301   (fraction of pairs with diff > 0 over all pairs)

Wilcoxon statistic : 43925.5
One-sided p-value  : 4.13828e-13   (H_A: real > fake)
Effect size r_rb   : 0.430   (rank-biserial correlation; -1..+1, >0 favors real)
Decision (alpha=0.01): REJECT H0


In [6]:
# --- Step 9: read confirmed neuron IDs (from your labels CSV) ---
import csv
import numpy as np
import os  

def _candidate_fake_keys(orig_fname):
    """
    Build possible keys for the fake CSV when fakes are named like 'fake_<original basename>.jpg'.
    Tries (in order):
      1) same dir, prefixed basename: 'dir/fake_name.jpg'
      2) whole relpath prefixed:      'fake_dir/name.jpg'
      3) just prefixed basename:      'fake_name.jpg'
      4) exact original:              'dir/name.jpg'
    """
    d = os.path.dirname(orig_fname)
    b = os.path.basename(orig_fname)
    pref_b = f"fake_{b}"

    cand = []
    cand.append(os.path.join(d, pref_b) if d else pref_b)
    cand.append(f"fake_{orig_fname}")
    cand.append(pref_b)
    cand.append(orig_fname)

    seen = set(); out = []
    for c in cand:
        if c not in seen:
            seen.add(c); out.append(c)
    return out


def load_confirmed_neuron_ids(confirmed_csv_path, id_col_candidates=("neuron_id","neuron","id")):
    """
    Returns a sorted list of unique confirmed neuron IDs (ints in [0,63]).
    """
    with open(confirmed_csv_path, "r", newline="", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        header = rdr.fieldnames or []

        # find the ID column
        id_col = None
        for c in id_col_candidates:
            if c in header:
                id_col = c
                break
        if id_col is None:
            raise RuntimeError(f"No neuron-id column found; looked for {id_col_candidates}")

        ids = []
        for row in rdr:
            try:
                j = int(float(row[id_col]))  # handle "12.0" too
                if 0 <= j <= 63:
                    ids.append(j)
            except Exception:
                # skip bad rows
                continue

        ids = sorted(set(ids))
        if not ids:
            raise RuntimeError("No valid confirmed neuron IDs (0..63) found.")
        return ids


In [7]:
# --- Step 10: confirmed-only fired counts using the 0.8× thresholds from REAL ---

def write_confirmed_fired_counts(real_csv_path,
                                 fake_csv_path,
                                 confirmed_csv_path,
                                 out_csv_path,
                                 ratio=0.8,
                                 error_list=None):
    """
    Reads:
      - real_csv_path: 64-neuron activations for real images
      - fake_csv_path: 64-neuron activations for fake images
      - confirmed_csv_path: subset of neuron IDs that are "confirmed" (with labels)
    Writes:
      - out_csv_path with columns: filenames, real_confirmed_count, fake_confirmed_count, total_confirmed_neurons
    """

    # Reuse helpers from Step 7; if you didn't run them, paste them above:
    #   load_csv_as_dicts, find_filename_column, ensure_neuron_columns,
    #   compute_threshold_from_real, count_fired

    # 1) load CSVs
    real_header, real_rows = load_csv_as_dicts(real_csv_path)
    fake_header, fake_rows = load_csv_as_dicts(fake_csv_path)

    real_fname_col = find_filename_column(real_header)
    fake_fname_col = find_filename_column(fake_header)

    neuron_cols = ensure_neuron_columns(real_header, n=64)

    # 2) thresholds from REAL (0.8× per-neuron max)
    threshold = compute_threshold_from_real(real_rows, neuron_cols, ratio=ratio)  # (64,)

    # 3) confirmed neuron IDs
    confirmed_ids = load_confirmed_neuron_ids(confirmed_csv_path)  # e.g., [3,7,12,...]
    confirmed_mask = np.zeros(64, dtype=bool)
    confirmed_mask[confirmed_ids] = True
    total_confirmed = int(np.sum(confirmed_mask))

        # 4) index fake rows by filename
    fake_idx = { r[fake_fname_col]: r for r in fake_rows }

    # 5) write counts
    with open(out_csv_path, "w", newline="", encoding="utf-8") as f_out:
        w = csv.writer(f_out)
        w.writerow(["filenames", "real_confirmed_count", "fake_confirmed_count", "total_confirmed_neurons"])

        for r in real_rows:
            fname = r[real_fname_col]

            # real confirmed count
            try:
                real_vec = np.array([float(r[c]) for c in neuron_cols], dtype=np.float64)
                real_count = int(np.sum(real_vec[confirmed_mask] >= threshold[confirmed_mask]))
            except Exception as e:
                if error_list is not None:
                    error_list.append({"stage":"count-real-confirmed","item":fname,"error":f"{type(e).__name__}: {e}"})
                real_count = ""

            # fake confirmed count (try fake_ prefix variants)
            try:
                fr = None
                for key in _candidate_fake_keys(fname):
                    fr = fake_idx.get(key)
                    if fr is not None:
                        break

                if fr is None:
                    if error_list is not None:
                        error_list.append({
                            "stage":"match-fake-confirmed",
                            "item":fname,
                            "error":"FileNotFoundError: fake row not found (tried fake_<basename> in dir, fake_<relpath>, basename, original)"
                        })
                    fake_count = ""
                else:
                    fake_vec = np.array([float(fr[c]) for c in neuron_cols], dtype=np.float64)
                    fake_count = int(np.sum(fake_vec[confirmed_mask] >= threshold[confirmed_mask]))
            except Exception as e:
                if error_list is not None:
                    error_list.append({"stage":"count-fake-confirmed","item":fname,"error":f"{type(e).__name__}: {e}"})
                fake_count = ""

            w.writerow([fname, real_count, fake_count, total_confirmed])



In [8]:
# --- Step 11: run confirmed-only counts ---

REAL_ACT_CSV_PATH = "preds_of_64Neurons_denseLayer_training.csv"      # your real activations (64 cols + filenames)
FAKE_ACT_CSV_PATH = "[img+obj_labels_to_fakes]fake_activations.csv"                            # produced earlier
CONFIRMED_CSV_PATH = "verification_combine_d(summary_1).csv"  # your uploaded file
CONFIRMED_COUNTS_OUT = "[img+obj_labels_to_fakes]fired_counts_confirmed_real_vs_fake.csv"      # new output

# Reuse shared ERRORS list if you want:
try:
    write_confirmed_fired_counts(
        real_csv_path=REAL_ACT_CSV_PATH,
        fake_csv_path=FAKE_ACT_CSV_PATH,
        confirmed_csv_path=CONFIRMED_CSV_PATH,
        out_csv_path=CONFIRMED_COUNTS_OUT,
        ratio=0.8,
        error_list=ERRORS  # optional
    )
except Exception as e:
    log_error("confirmed-counts", "aggregate", e)

# Optional: persist errors
if ERRORS:
    with open("pipeline_errors.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["stage","item","error"])
        w.writeheader(); w.writerows(ERRORS)
    print(f"[errors] wrote pipeline_errors.csv with {len(ERRORS)} entries")
else:
    print("[ok] no errors logged")


[ok] no errors logged


In [10]:
# for confirmed neurons only

# Wilcoxon (one-sided, paired) for: H_A = real_confirmed_count > fake_confirmed_count
# Requires: pip install scipy

import csv
import numpy as np
from scipy.stats import wilcoxon, rankdata

CSV_PATH = "[img+obj_labels_to_fakes]fired_counts_confirmed_real_vs_fake.csv"  # filenames, real_confirmed_count, fake_confirmed_count

# --- load paired counts ---
reals, fakes = [], []
with open(CSV_PATH, "r", newline="", encoding="utf-8") as f:
    rdr = csv.DictReader(f)
    for r in rdr:
        try:
            rv = float(r["real_confirmed_count"])
            fv = float(r["fake_confirmed_count"])
        except Exception:
            continue
        if np.isnan(rv) or np.isnan(fv):
            continue
        reals.append(rv); fakes.append(fv)

reals = np.asarray(reals, dtype=float)
fakes = np.asarray(fakes, dtype=float)
if reals.size == 0:
    raise RuntimeError("No valid (real,fake) pairs found in CSV. Check column names/values.")

diffs = reals - fakes  # test if diffs > 0  (i.e., fake < real)

# --- run Wilcoxon (zeros dropped by 'zero_method=wilcox') ---
stat, p = wilcoxon(diffs, alternative="greater", zero_method="wilcox", correction=False, mode="auto")

# --- descriptive stats ---
n_total   = diffs.size
nz_mask   = diffs != 0
n_nonzero = int(nz_mask.sum())
n_zero    = n_total - n_nonzero

med_real  = float(np.median(reals))
med_fake  = float(np.median(fakes))
med_diff  = float(np.median(diffs))
mean_diff = float(np.mean(diffs))
prop_pos  = float(np.mean(diffs > 0.0))  # proportion of pairs with real > fake (over all pairs)

# --- effect size: rank-biserial correlation (r_rb), on nonzero diffs only ---
if n_nonzero > 0:
    d_nz = diffs[nz_mask]
    ranks = rankdata(np.abs(d_nz), method="average")       # ranks of absolute diffs
    W_plus  = ranks[d_nz > 0].sum()
    W_minus = ranks[d_nz < 0].sum()
    denom = n_nonzero * (n_nonzero + 1) / 2.0
    r_rb = (W_plus - W_minus) / denom if denom > 0 else np.nan
else:
    r_rb = np.nan

# --- print report ---
print("=== Wilcoxon Signed-Rank Test (one-sided, real > fake) ===")
print(f"File: {CSV_PATH}")
print(f"N total pairs      : {n_total}")
print(f"N nonzero pairs    : {n_nonzero} (pairs used by Wilcoxon)")
print(f"N zero differences : {n_zero}")
print()
print(f"Median(real)       : {med_real:.3f}")
print(f"Median(fake)       : {med_fake:.3f}")
print(f"Median(diff)       : {med_diff:.3f}   (diff = real - fake)")
print(f"Mean(diff)         : {mean_diff:.3f}")
print(f"P(real > fake)     : {prop_pos:.3f}   (fraction of pairs with diff > 0 over all pairs)")
print()
print(f"Wilcoxon statistic : {stat:.6g}")
print(f"One-sided p-value  : {p:.6g}   (H_A: real > fake)")
print(f"Effect size r_rb   : {r_rb:.3f}   (rank-biserial correlation; -1..+1, >0 favors real)")
print("Decision (alpha=0.01): " + ("REJECT H0" if p < 0.01 else "fail to reject H0"))


=== Wilcoxon Signed-Rank Test (one-sided, real > fake) ===
File: [img+obj_labels_to_fakes]fired_counts_confirmed_real_vs_fake.csv
N total pairs      : 3157
N nonzero pairs    : 562 (pairs used by Wilcoxon)
N zero differences : 2595

Median(real)       : 0.000
Median(fake)       : 0.000
Median(diff)       : 0.000   (diff = real - fake)
Mean(diff)         : 0.050
P(real > fake)     : 0.107   (fraction of pairs with diff > 0 over all pairs)

Wilcoxon statistic : 96160
One-sided p-value  : 1.09972e-06   (H_A: real > fake)
Effect size r_rb   : 0.216   (rank-biserial correlation; -1..+1, >0 favors real)
Decision (alpha=0.01): REJECT H0
